# Lesson 4.3: Conditional edges: your loop as a graph

**Goal:** rebuild the ReAct loop of lesson 3.1 as a LangGraph with a `model` node, a `tools` node and one conditional edge.

**What you will be able to do**
- Write a `route` function that picks `tools`, `model` (retry) or `END` from the last message.
- Build, draw (Mermaid) and stream a two-node graph with a reducer on `messages`.
- Map every piece of your `react()` loop to a piece of the graph.
- Predict where `recursion_limit` stops a run, and catch `GraphRecursionError`.

In [1]:
# Setup: find the course repo's nightwatch/ folder (relative to this notebook) and load .env if present.
import os, sys
from pathlib import Path

for up in ("../../nightwatch", "../../../nightwatch", "../nightwatch", "nightwatch"):
    if Path(up, "brightcart").is_dir():
        NW = Path(up)
        break
else:
    raise FileNotFoundError("Open this notebook from <lesson>/notebook inside the course repo: it needs the nightwatch/ folder.")
sys.path.insert(0, str(NW.resolve()))

try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))      # walks up from here and loads the repo's .env
except ImportError:
    pass
HAS_KEY = bool(os.getenv("GROQ_API_KEY"))
print("nightwatch folder:", NW.as_posix(), "| GROQ_API_KEY:", "found" if HAS_KEY else "not set")
if not HAS_KEY:
    print("No GROQ_API_KEY: get the free key as shown in lesson 0.2, put GROQ_API_KEY=... in the repo's .env,")
    print("then restart. Cells that need it: the cells marked 'calls the API' (4 cells: the graph run, react(), recursion_limit=4, the limit-10 run in Exercise 1's extra cell). They replay from the cache when it has the answer.")

nightwatch folder: ../../../nightwatch | GROQ_API_KEY: found


## 1. Recall: reducers (lesson 4.2)

A reducer joins a node's update to the old value; a key without one keeps only the last write. Today the reducer sits
on the message list, so each node returns only its **new** messages.

In [2]:
import operator
print(operator.add([{"role": "system"}, {"role": "user"}], [{"role": "assistant"}]))   # what the reducer does

[{'role': 'system'}, {'role': 'user'}, {'role': 'assistant'}]


## 2. Key words

| Word | Meaning here |
|---|---|
| **conditional edge** | a function run after a node; it returns the next node's name |
| **END** | the special stop node: reaching it finishes the run |
| **model node / tools node** | one model call / runs every tool the last message asked for |
| **Mermaid** | a small text format for graphs; LangGraph writes it for you |
| **recursion limit** | caps node runs per call; past it LangGraph raises `GraphRecursionError` |

## 3. The idea: the loop, drawn as a graph

Think = one model call = the `model` node. Act + observe = the `tools` node. From `tools` there is one way back to
`model` (a plain edge). From `model` there are choices, picked by `route`:

| last message | route returns |
|---|---|
| has `tool_calls` | `"tools"` |
| a `user` message (our retry note after a malformed call) | `"model"` |
| anything else (an answer) | `END` |

**Hand trace: `upstream_deploy-0`**, the state starts with 2 messages (system rules + pager alert):

| step | node | what it adds | messages after |
|---|---|---|---|
| 1 | model | asks check_health | 3 |
| 2 | tools | checkout and payments unhealthy | 4 |
| 3 | model | asks recent_deploys | 5 |
| 4 | tools | payments v1.3.5 at 04:12 | 6 |
| 5 | model | asks read_logs(checkout) | 7 |
| 6 | tools | "upstream payments returned 500" | 8 |
| 7 | model | asks rollback_deploy(payments) | 9 |
| 8 | tools | "rollback_deploy(payments) done." | 10 |
| 9 | model | asks check_health | 11 |
| 10 | tools | healthy: true | 12 |
| 11 | model | answer, no tool call, route says END | 13 |

In total: 6 model calls, 5 tool runs, 1 write, and a healthy shop.

## 4. Shared setup: incident, prompt, the same four tools as lesson 3.1

In [3]:
import json
from typing import Annotated, TypedDict

import agent_v0                                   # chat(): record/replay cache, so reruns are free
from brightcart.incidents import fresh, generate
from brightcart.sim import World

INC = next(x for x in generate() if x.id == "upstream_deploy-0")
SYSTEM = "You are Nightwatch, Brightcart's on-call agent. Fix the problem, check health, then answer."

def tool(name, desc, **props):
    return {"type": "function", "function": {"name": name, "description": desc, "parameters": {
        "type": "object", "properties": {k: {"type": "string"} for k in props}, "required": list(props)}}}

TOOLS = [tool("read_logs", "Last log lines of a service.", service=1),
         tool("recent_deploys", "Deploys in the last hour."),
         tool("rollback_deploy", "Roll back the latest deploy of a service.", service=1),
         tool("check_health", "Is every service healthy?")]

def tool_runner(world):
    return {"read_logs": lambda a: world.logs(a["service"], 8), "recent_deploys": lambda a: world.inc.deploys,
            "rollback_deploy": lambda a: world.act("rollback_deploy", a["service"]),
            "check_health": lambda a: world.health()}

start = {"messages": [{"role": "system", "content": SYSTEM}, {"role": "user", "content": f"PAGER: {INC.alert}"}]}
print(INC.id, "|", start["messages"][1]["content"])

upstream_deploy-0 | PAGER: checkout error rate above 5%


## 5. Write it yourself: `react()` from lesson 3.1

The loop you wrote in lesson 3.1, with one change: the model call goes through `agent_v0.chat` (record/replay) instead
of the OpenAI client, so this cell replays from the cache. Five pieces: the message list, the loop, the model call,
the answer check, the tool runs, and the step limit at the bottom that *returns a string*.

**Calls the API** (replays from the cache when the answers are recorded; then no key is needed).

In [4]:
def react(goal, world, max_steps=8):
    run = tool_runner(world)
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": goal}]
    for step in range(1, max_steps + 1):
        body = {"model": "openai/gpt-oss-20b", "messages": msgs, "tools": TOOLS, "temperature": 0}
        m = agent_v0.chat(body, mode="record")["choices"][0]["message"]
        if not m.get("tool_calls"):                            # no action asked -> this is the answer
            print(f"step {step}: ANSWER  {m['content'][:80]!r}")
            return m["content"]
        msgs.append({"role": "assistant", "content": m.get("content") or "", "tool_calls": m["tool_calls"]})
        for c in m["tool_calls"]:                              # act, observe, feed the result back
            args = json.loads(c["function"]["arguments"] or "{}")
            try:
                out = run[c["function"]["name"]](args)
            except Exception as e:
                out = f"ERROR: {e}"
            print(f"step {step}: {c['function']['name']}({args}) -> {str(out)[:70]}")
            msgs.append({"role": "tool", "tool_call_id": c["id"], "content": json.dumps(out, default=str)})
    return "stopped: step limit"

w_react = World(fresh(INC))
answer = react(f"PAGER: {INC.alert}", w_react)
print("healthy:", w_react.health()["healthy"], "| writes:", w_react.actions)

step 1: check_health({'': ''}) -> {'healthy': False, 'unhealthy_services': ['checkout', 'payments'], 'qu
step 2: recent_deploys({'': ''}) -> [{'t': '04:12', 'service': 'payments', 'version': 'v1.3.5', 'by': 'ci'
step 3: read_logs({'service': 'checkout'}) -> ['04:10 WARN checkout slow GC pause 120ms', '04:13 ERROR checkout upst
step 4: rollback_deploy({'service': 'payments'}) -> rollback_deploy(payments) done.
step 5: check_health({'': ''}) -> {'healthy': True, 'unhealthy_services': [], 'queues_over_limit': {}}
step 6: ANSWER  '**Action taken**\n\n1. **Identified the issue** – The `checkout` service was repor'
healthy: True | writes: [('rollback_deploy', 'payments', None)]


In the video, the lesson 3.1 run (OpenAI client, live) took **8** model calls: it rolled back before reading any logs and
read the payments logs twice. Here, `react()` sends the very same requests as the graph below, so it replays the
graph's recorded replies and takes the graph's 6-call path. Same requests in, same replies out: that is the replay
idea lesson 4.5 uses to *prove* two agents are the same.

## 6. The library version: the same loop in LangGraph

The state has one key, `messages`, with `operator.add` as its reducer. The model node is one turn of the old loop;
the tools node is the inner for loop; `route` is the answer check.

In [5]:
from langgraph.errors import GraphRecursionError
from langgraph.graph import END, START, StateGraph

class AgentState(TypedDict):
    messages: Annotated[list, operator.add]     # OpenAI-style dicts, appended

def route(state):                                # = "if not m.tool_calls: return"
    last = state["messages"][-1]
    return "tools" if last.get("tool_calls") else ("model" if last["role"] == "user" else END)

def make_graph(world, route_fn=route):
    run = tool_runner(world)

    def model(state):                            # = one turn of the loop
        body = {"model": "openai/gpt-oss-20b", "messages": state["messages"], "tools": TOOLS, "temperature": 0}
        out = agent_v0.chat(body, mode="record")
        if "error" in out:                       # Groq rejected a malformed tool call: ask again
            return {"messages": [{"role": "user", "content": "Your last tool call was malformed. Try again."}]}
        m = out["choices"][0]["message"]
        msg = {"role": "assistant", "content": m.get("content") or ""}
        if m.get("tool_calls"):
            msg["tool_calls"] = m["tool_calls"]
        return {"messages": [msg]}

    def tools(state):                            # = the for loop over tool calls
        outs = []
        for c in state["messages"][-1]["tool_calls"]:
            args = json.loads(c["function"]["arguments"] or "{}")
            try:
                res = run[c["function"]["name"]](args)
            except Exception as e:
                res = f"ERROR: {e}"
            outs.append({"role": "tool", "tool_call_id": c["id"], "content": json.dumps(res, default=str)})
        return {"messages": outs}

    g = StateGraph(AgentState)
    g.add_node("model", model)
    g.add_node("tools", tools)
    g.add_edge(START, "model")
    g.add_conditional_edges("model", route_fn, ["tools", "model", END])
    g.add_edge("tools", "model")
    return g.compile()

world = World(fresh(INC))
graph = make_graph(world)
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	model(model)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> model;
	model -.-> __end__;
	model -.-> tools;
	tools --> model;
	model -.-> model;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



Solid arrows are plain edges; dotted arrows are the conditional ones. The dotted `model -.-> model` is the retry path.

**Calls the API** (replays from the cache): stream the run node by node.

In [6]:
UPDATES = []                                     # kept for Exercise 2
for i, chunk in enumerate(graph.stream(start, stream_mode="updates", config={"recursion_limit": 25}), 1):
    (node, upd), = chunk.items()
    UPDATES.append((node, upd))
    for m in upd["messages"]:
        if m["role"] == "assistant" and m.get("tool_calls"):
            what = ", ".join(f"{c['function']['name']}({c['function']['arguments']})" for c in m["tool_calls"])
            print(f"{i:2d} {node:6} asks {what}")
        elif m["role"] == "tool":
            print(f"{i:2d} {node:6} -> {m['content'][:80]}")
        else:
            print(f"{i:2d} {node:6} {m['role']}: {m['content'][:80]!r}")
print(f"\nhealthy {world.health()['healthy']} | writes {world.actions}")

 1 model  asks check_health({"":""})
 2 tools  -> {"healthy": false, "unhealthy_services": ["checkout", "payments"], "queues_over_
 3 model  asks recent_deploys({"":""})
 4 tools  -> [{"t": "04:12", "service": "payments", "version": "v1.3.5", "by": "ci"}, {"t": "
 5 model  asks read_logs({"service":"checkout"})
 6 tools  -> ["04:10 WARN checkout slow GC pause 120ms", "04:13 ERROR checkout upstream payme
 7 model  asks rollback_deploy({"service":"payments"})
 8 tools  -> "rollback_deploy(payments) done."
 9 model  asks check_health({"":""})
10 tools  -> {"healthy": true, "unhealthy_services": [], "queues_over_limit": {}}
11 model  assistant: '**Action taken**\n\n1. **Identified the issue** – The `checkout` service was repor'

healthy True | writes [('rollback_deploy', 'payments', None)]


**react() vs the graph, piece by piece**

| react() (lesson 3.1) | the graph |
|---|---|
| `msgs` list | the state, one key `messages` with the `add` reducer |
| `for step in range(...)` | no line of its own: the edge `tools -> model` |
| the model call | the `model` node |
| `if not m.tool_calls: return` | `route`, and its arrow to `END` |
| `msgs.append(assistant)` | the model node's return value |
| inner loop over tool calls | the `tools` node, almost line for line |
| `max_steps` (returns a string) | `recursion_limit` (raises `GraphRecursionError`) |

Added by the graph: a printable drawing, streamed updates, a built-in step limit, and the retry path as a visible edge.
It is longer (about 30 lines against 21), partly for that retry branch.

### A step limit you don't write: `recursion_limit = 4`

It counts **node runs**: model, tools, model, tools, then the error. The rollback came at steps 7 and 8, so nothing is
written. **Calls the API** (replays from the cache).

In [7]:
w2 = World(fresh(INC))
try:
    make_graph(w2).invoke(start, config={"recursion_limit": 4})
except GraphRecursionError as e:
    print("GraphRecursionError:", str(e).splitlines()[0])
    print(f"world after the stop: healthy {w2.health()['healthy']} | writes {w2.actions}")

GraphRecursionError: Recursion limit of 4 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.
world after the stop: healthy False | writes []


## 7. Where this shows up in agents

This two-node graph *is* the agent loop: every LangGraph agent you meet, including `create_agent` in lesson 4.4, is this
model node, a tools node and the `route` decision, plus middleware around it. Here it runs with a scripted fake model,
so you can test routing and wiring with no API at all:

In [8]:
def fake_model(script):
    """A model node that replays a fixed list of replies: a tool call, or a plain answer."""
    replies = iter(script)
    def model(state):
        r = next(replies)
        if r == "retry":
            return {"messages": [{"role": "user", "content": "Your last tool call was malformed. Try again."}]}
        if r.startswith("call:"):
            name = r[5:]
            return {"messages": [{"role": "assistant", "content": "",
                                  "tool_calls": [{"id": f"c{len(state['messages'])}", "type": "function",
                                                  "function": {"name": name, "arguments": "{}"}}]}]}
        return {"messages": [{"role": "assistant", "content": r}]}
    return model

def fake_tools(state):
    return {"messages": [{"role": "tool", "tool_call_id": c["id"], "content": f"ran {c['function']['name']}"}
                         for c in state["messages"][-1]["tool_calls"]]}

def toy_graph(script, route_fn=route, edge_back=True):
    g = StateGraph(AgentState)
    g.add_node("model", fake_model(script))
    g.add_node("tools", fake_tools)
    g.add_edge(START, "model")
    g.add_conditional_edges("model", route_fn, ["tools", "model", END])
    if edge_back:
        g.add_edge("tools", "model")
    return g.compile()

out = toy_graph(["call:check_health", "retry", "call:check_health", "all healthy"]).invoke({"messages": []})
print([m["role"] for m in out["messages"]])
print("answer:", out["messages"][-1]["content"])

['assistant', 'tool', 'user', 'assistant', 'tool', 'assistant']
answer: all healthy


## 8. Traps
Each trap runs without stopping the notebook (no API).

**Trap 1: reading `recursion_limit` as loop steps.** It counts node runs, so one loop step (model + tools) costs two.

In [9]:
seen = []
try:
    for chunk in toy_graph(["call:check_health"] * 10).stream({"messages": []}, stream_mode="updates",
                                                              config={"recursion_limit": 4}):
        seen += list(chunk)
except GraphRecursionError as e:
    print("nodes run:", seen, "->", type(e).__name__)

nodes run: ['model', 'tools', 'model', 'tools'] -> GraphRecursionError


**Trap 2: a route that only checks for tool calls.** The retry note is a user message with no tool calls, so it ends the run instead of asking again.

In [10]:
def route_only_tools(state):
    return "tools" if state["messages"][-1].get("tool_calls") else END

out = toy_graph(["call:check_health", "retry", "call:check_health", "all healthy"],
                route_fn=route_only_tools).invoke({"messages": []})
print("last message:", out["messages"][-1]["role"], "|", out["messages"][-1]["content"])

last message: user | Your last tool call was malformed. Try again.


**Trap 3: forgetting the edge from tools back to model.** It compiles, but the run ends after `tools`: the tool result never reaches the model.

In [11]:
out = toy_graph(["call:check_health", "all healthy"], edge_back=False).invoke({"messages": []})
print([m["role"] for m in out["messages"]], "| no answer:", out["messages"][-1]["role"] != "assistant")

['assistant', 'tool'] | no answer: True


**Trap 4: trusting one run to show two agents are the same.** Same tools, prompt, incident and temperature 0, yet the live lesson 3.1 run took 8 model calls and the graph took 6. At temperature 0 the path can still change (lesson 1.2: 8 identical requests gave 2 wordings). Showing sameness needs replay of recorded replies (lesson 4.5).

In [12]:
path_31 = ["check_health", "recent_deploys", "rollback_deploy", "read_logs", "read_logs", "read_logs", "check_health"]
path_43 = [c["function"]["name"] for n, u in UPDATES for m in u["messages"] for c in m.get("tool_calls", [])]
print("lesson 3.1 run:", len(path_31) + 1, "model calls |", path_31)
print("graph run     :", len(path_43) + 1, "model calls |", path_43)
print("same final write in both: rollback_deploy(payments)")

lesson 3.1 run: 8 model calls | ['check_health', 'recent_deploys', 'rollback_deploy', 'read_logs', 'read_logs', 'read_logs', 'check_health']
graph run     : 6 model calls | ['check_health', 'recent_deploys', 'read_logs', 'rollback_deploy', 'check_health']
same final write in both: rollback_deploy(payments)


## 9. Exercises
Each exercise has a starter cell (`# YOUR CODE HERE`) and a check cell. The checks need no API.

### Exercise 1 (the video's exercise): print the decision, predict the last node

Write `route_logged(state)`: same decision as `route`, but it also prints it (e.g. `route -> tools`) and appends it
to the list `DECISIONS`. Then set `predicted_last` to the node you expect to run **last** before the
`GraphRecursionError` when the limit is 10 (`"model"` or `"tools"`).

In [13]:
DECISIONS = []

def route_logged(state):
    # YOUR CODE HERE
    return END

predicted_last = None   # "model" or "tools"

In [14]:
try:
    DECISIONS.clear()
    assert route_logged({"messages": [{"role": "assistant", "content": "", "tool_calls": [{"id": "x"}]}]}) == "tools", "tool calls in the last message mean tools"
    assert route_logged({"messages": [{"role": "user", "content": "retry"}]}) == "model", "a user note means retry"
    assert route_logged({"messages": [{"role": "assistant", "content": "done"}]}) == END
    assert DECISIONS == ["tools", "model", END], f"DECISIONS should log every decision, got {DECISIONS}"
    seen = []
    try:
        for ch in toy_graph(["call:check_health"] * 20, route_fn=route_logged).stream(
                {"messages": []}, stream_mode="updates", config={"recursion_limit": 10}):
            seen += list(ch)
    except GraphRecursionError:
        pass
    assert predicted_last == seen[-1], f"you predicted {predicted_last!r}; the last node run was {seen[-1]!r}"
    print("✓ Exercise 1 passed")
except AssertionError as e:
    print("✗ Exercise 1 not yet:", e)

✗ Exercise 1 not yet: tool calls in the last message mean tools


Optional, **calls the API** (replays from the cache): the real model with `route_logged` and a limit of 10. Run it after Exercise 1 passes.

In [15]:
if predicted_last:
    w3 = World(fresh(INC))
    try:
        make_graph(w3, route_fn=route_logged).invoke(start, config={"recursion_limit": 10})
    except GraphRecursionError as e:
        print("GraphRecursionError | writes:", w3.actions)
else:
    print("Solve Exercise 1 first.")

Solve Exercise 1 first.


### Exercise 2: count the run from its updates

Write `count_run(updates)` for the `UPDATES` list saved in section 6 (pairs of node name and update). Return
`(model_calls, tool_runs, writes)`, where a write is a tool call named `rollback_deploy`.

In [16]:
def count_run(updates):
    # YOUR CODE HERE
    return 0, 0, 0

In [17]:
try:
    got = count_run(UPDATES)
    assert got == (6, 5, 1), f"expected (6, 5, 1), got {got}"
    assert count_run([]) == (0, 0, 0)
    print("✓ Exercise 2 passed")
except AssertionError as e:
    print("✗ Exercise 2 not yet:", e)

✗ Exercise 2 not yet: expected (6, 5, 1), got (0, 0, 0)


### Exercise 3: model calls before the error

The full run needs 11 node runs (6 model + 5 tools, alternating, starting with model). Write
`model_calls(limit)`: how many model calls run with that `recursion_limit` (all 6 if the run fits).

In [18]:
def model_calls(limit, full_run=11):
    # YOUR CODE HERE
    return 0

In [19]:
try:
    for lim, want in [(4, 2), (6, 3), (10, 5), (1, 1), (25, 6), (11, 6)]:
        assert model_calls(lim) == want, f"model_calls({lim}) should be {want}, got {model_calls(lim)}"
    print("✓ Exercise 3 passed")
except AssertionError as e:
    print("✗ Exercise 3 not yet:", e)

✗ Exercise 3 not yet: model_calls(4) should be 2, got 0


### Exercise 4: wire it yourself

Build `my_graph(script)` from scratch with `StateGraph(AgentState)`: nodes `model` (use `fake_model(script)`) and
`tools` (`fake_tools`), START to model, the conditional edge with `route`, and the edge back. Compile and return it.

In [20]:
def my_graph(script):
    # YOUR CODE HERE
    return None

In [21]:
try:
    g = my_graph(["call:recent_deploys", "call:check_health", "fixed"])
    assert g is not None, "return the compiled graph"
    out = g.invoke({"messages": [{"role": "user", "content": "PAGER"}]})
    roles = [m["role"] for m in out["messages"]]
    assert roles == ["user", "assistant", "tool", "assistant", "tool", "assistant"], f"got {roles}"
    assert out["messages"][-1]["content"] == "fixed"
    print("✓ Exercise 4 passed")
except AssertionError as e:
    print("✗ Exercise 4 not yet:", e)

✗ Exercise 4 not yet: return the compiled graph


## Solutions (try first!)

In [22]:
DECISIONS = []

def route_logged(state):
    d = route(state)
    print("route ->", d)
    DECISIONS.append(d)
    return d

predicted_last = "tools"   # 10 node runs: model, tools x 5, so tools runs last

In [23]:
def count_run(updates):
    model_calls = sum(node == "model" for node, _ in updates)
    tool_runs = sum(node == "tools" for node, _ in updates)
    writes = sum(c["function"]["name"] == "rollback_deploy"
                 for _, u in updates for m in u["messages"] for c in m.get("tool_calls", []))
    return model_calls, tool_runs, writes

In [24]:
def model_calls(limit, full_run=11):
    runs = min(limit, full_run)
    return (runs + 1) // 2          # model runs at node runs 1, 3, 5, ...

In [25]:
def my_graph(script):
    g = StateGraph(AgentState)
    g.add_node("model", fake_model(script))
    g.add_node("tools", fake_tools)
    g.add_edge(START, "model")
    g.add_conditional_edges("model", route, ["tools", "model", END])
    g.add_edge("tools", "model")
    return g.compile()